In [1]:
import pandas as pd
import requests

In [ ]:
# "NRT" stands for Near Real-Time, meaning "recent days only."

In [ ]:
##FIRMS NASA FIRE  DATA PULLING FOR 202-2024(APRIL,MAY,JUNE)

In [11]:
MAP_KEY="2cdc83f40fbd7a1a9fe88b50922d1218"    #personal key to req data from nasa firms
area = "78.5,29.0,80.0,30.6"        # always remeber with no GAPS,BCZ  IF NOT MIGHT lead to error in undertanding by nas

In [16]:
#f"..." → this is called an "f-string" — a way to build a sentence that includes our variables inside curly brace
url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/VIIRS_SNPP_NRT/{area}/5"
response = requests.get(url)
print(response.status_code)
print(url)

200
https://firms.modaps.eosdis.nasa.gov/api/area/csv/2cdc83f40fbd7a1a9fe88b50922d1218/VIIRS_SNPP_NRT/78.5,29.0,80.0,30.6/5


In [18]:
print(response.text)

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight


In [19]:
print(area)
print(MAP_KEY)

78.5,29.0,80.0,30.6
2cdc83f40fbd7a1a9fe88b50922d1218


In [20]:
import io
df = pd.read_csv(io.StringIO(response.text))   ##IO CONVERTS CSV FORM INTO STRING FORM
print(df.shape)
df.head()

(0, 14)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight


In [23]:
test_date = "2024-04-20"
url_historical = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/VIIRS_SNPP_SP/{area}/5/{test_date}"
response2 = requests.get(url_historical)
print(response2.status_code)

200


In [24]:
print(response2.text)

latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
29.11195,79.85496,338.14,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,306.51,3.69,D,0
29.11302,79.81827,341.36,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,309.6,5.76,D,0
29.1132,79.86288,335.71,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,309.18,2.73,D,0
29.11382,79.86684,339.64,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,308.89,2.73,D,0
29.11527,79.85435,333.36,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,304.72,3.55,D,0
29.11632,79.81759,348.64,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,307.02,4.04,D,0
29.11776,79.87019,336.61,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,305.24,2.35,D,0
29.11819,79.82946,332.34,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,303.71,2.27,D,0
29.11837,79.87411,343.63,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,306.04,2.35,D,0
29.11962,79.8169,338.63,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,302.95,4.04,D,0
29.12024,79.82085,330.23,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,303.03,4.49,D,0
29.12087,79.82482

In [25]:
df_historical = pd.read_csv(io.StringIO(response2.text))
print(df_historical.shape)
df_historical.head()

(1323, 15)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,29.11195,79.85496,338.14,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,306.51,3.69,D,0
1,29.11302,79.81827,341.36,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,309.60,5.76,D,0
2,29.11320,79.86288,335.71,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,309.18,2.73,D,0
3,29.11382,79.86684,339.64,0.39,0.36,2024-04-20,743,N,VIIRS,n,2,308.89,2.73,D,0
4,29.11527,79.85435,333.36,0.39,0.36,2024-04-20,743,N,VIIRS,l,2,304.72,3.55,D,0


In [27]:
df_historical.shape

(1323, 15)

In [28]:
df_historical.to_csv("../data/raw/firms_2024-04-20_sample.csv", index=False)   #STORING DATAFRAMES AS IN CSV FILE IN OUR FILE ADDRESS
print("Saved!")

Saved!


In [29]:
import datetime

start = datetime.date(2024, 4, 1)
end = datetime.date(2024, 6, 30)

dates_to_pull = []
current = start
while current <= end:
    dates_to_pull.append(current.strftime("%Y-%m-%d"))
    current += datetime.timedelta(days=5)

print(len(dates_to_pull))
print(dates_to_pull[:5])

19
['2024-04-01', '2024-04-06', '2024-04-11', '2024-04-16', '2024-04-21']


In [30]:
import time

all_data = []

for date in dates_to_pull:
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/VIIRS_SNPP_SP/{area}/5/{date}"
    response = requests.get(url)
    
    if response.status_code == 200:
        temp_df = pd.read_csv(io.StringIO(response.text))
        all_data.append(temp_df)
        print(date, "->", temp_df.shape[0], "fires found")
    else:
        print(date, "-> FAILED, status code:", response.status_code)
    
    time.sleep(1)

2024-04-01 -> 241 fires found
2024-04-06 -> 1043 fires found
2024-04-11 -> 253 fires found
2024-04-16 -> 986 fires found
2024-04-21 -> 1718 fires found
2024-04-26 -> 1576 fires found
2024-05-01 -> 3641 fires found
2024-05-06 -> 605 fires found
2024-05-11 -> 150 fires found
2024-05-16 -> 315 fires found
2024-05-21 -> 67 fires found
2024-05-26 -> 86 fires found
2024-05-31 -> 6 fires found
2024-06-05 -> 163 fires found
2024-06-10 -> 846 fires found
2024-06-15 -> 567 fires found
2024-06-20 -> 9 fires found
2024-06-25 -> 1 fires found
2024-06-30 -> 0 fires found


In [31]:
df_season_2024 = pd.concat(all_data, ignore_index=True)
print(df_season_2024.shape)
df_season_2024.head()

(12273, 15)


C:\Users\dolly\AppData\Local\Temp\ipykernel_9432\3665869033.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_season_2024 = pd.concat(all_data, ignore_index=True)


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight,type
0,29.57618,79.35428,335.68,0.54,0.68,2024-04-01,840,N,VIIRS,n,2,294.81,2.68,D,0
1,29.57858,79.35174,327.62,0.54,0.68,2024-04-01,840,N,VIIRS,n,2,296.31,4.07,D,0
2,29.71142,79.29804,335.84,0.54,0.68,2024-04-01,840,N,VIIRS,n,2,291.65,4.72,D,0
3,29.06977,79.96082,295.96,0.35,0.56,2024-04-01,2104,N,VIIRS,n,2,283.83,0.74,N,0
4,29.10752,79.82458,301.99,0.34,0.56,2024-04-01,2104,N,VIIRS,n,2,288.11,0.58,N,0


In [32]:
df_season_2024.to_csv("../data/raw/firms_2024_season.csv", index=False)
print("Saved! Total fires:", df_season_2024.shape[0])

Saved! Total fires: 12273


In [33]:
years = [2020, 2021, 2022, 2023, 2024]
all_dates_multi_year = []

for year in years:
    start = datetime.date(year, 4, 1)
    end = datetime.date(year, 6, 30)
    current = start
    while current <= end:
        all_dates_multi_year.append(current.strftime("%Y-%m-%d"))
        current += datetime.timedelta(days=5)

print(len(all_dates_multi_year))
print(all_dates_multi_year[:3])
print(all_dates_multi_year[-3:])

95
['2020-04-01', '2020-04-06', '2020-04-11']
['2024-06-20', '2024-06-25', '2024-06-30']


In [35]:
def get_fires_for_date(date):  #1TIME CODE RUNS N NUMBER TIME 
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/VIIRS_SNPP_SP/{area}/5/{date}"
    response = requests.get(url)
    
    if response.status_code == 200:
        temp_df = pd.read_csv(io.StringIO(response.text))
        return temp_df
    else:
        print(date, "-> FAILED, status code:", response.status_code)
        return None

In [36]:
all_data_multi_year = []

for date in all_dates_multi_year:
    result = get_fires_for_date(date)
    if result is not None:
        all_data_multi_year.append(result)
        print(date, "->", result.shape[0], "fires found")
    time.sleep(1)

print("DONE. Total chunks collected:", len(all_data_multi_year))

2020-04-01 -> 11 fires found
2020-04-06 -> 16 fires found
2020-04-11 -> 11 fires found
2020-04-16 -> 17 fires found
2020-04-21 -> 21 fires found
2020-04-26 -> 11 fires found
2020-05-01 -> 23 fires found
2020-05-06 -> 8 fires found
2020-05-11 -> 23 fires found
2020-05-16 -> 96 fires found
2020-05-21 -> 108 fires found
2020-05-26 -> 88 fires found
2020-05-31 -> 1 fires found
2020-06-05 -> 3 fires found
2020-06-10 -> 11 fires found
2020-06-15 -> 8 fires found
2020-06-20 -> 6 fires found
2020-06-25 -> 3 fires found
2020-06-30 -> 3 fires found
2021-04-01 -> 3279 fires found
2021-04-06 -> 1140 fires found
2021-04-11 -> 2564 fires found
2021-04-16 -> 519 fires found
2021-04-21 -> 88 fires found
2021-04-26 -> 336 fires found
2021-05-01 -> 144 fires found
2021-05-06 -> 66 fires found
2021-05-11 -> 26 fires found
2021-05-16 -> 40 fires found
2021-05-21 -> 11 fires found
2021-05-26 -> 4 fires found
2021-05-31 -> 3 fires found
2021-06-05 -> 6 fires found
2021-06-10 -> 2 fires found
2021-06-15 -> 0

In [37]:
df_all_years = pd.concat(all_data_multi_year, ignore_index=True)   # CONCATINATES 2 DATAFRAMES STACKING ON TOP OF EACH ITHER 
print(df_all_years.shape)
df_all_years['acq_date'] = pd.to_datetime(df_all_years['acq_date'])
print(df_all_years['acq_date'].min(), "to", df_all_years['acq_date'].max())

(31983, 15)
2020-04-02 00:00:00 to 2024-06-25 00:00:00


C:\Users\dolly\AppData\Local\Temp\ipykernel_9432\698215915.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_all_years = pd.concat(all_data_multi_year, ignore_index=True)


In [38]:
df_all_years.to_csv("../data/raw/firms_2020_2024_season.csv", index=False)
print("Saved! Total rows:", df_all_years.shape[0])

Saved! Total rows: 31983


In [ ]:
##FIRMS NASA FIRE  DATA PULLED FOR 202-2024(APRIL,MAY,JUNE)

In [ ]:
##WEATHER DATA PULLING FOR 202-2024(APRIL,MAY,JUNE)